In [ ]:
# En Google Colab: monta tu Drive y ajusta las rutas de abajo.
# En entorno local: esta celda se puede omitir.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
import os
if 'IN_COLAB' in dir() and IN_COLAB:
    os.chdir("/content/drive/MyDrive/Courses/AI/masked_attention/llama_like/excercises")
# En entorno local el directorio de trabajo ya es el correcto.


In [ ]:
import sys, os
# Agrega el directorio padre (donde está src/) al path
parent = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent not in sys.path:
    sys.path.insert(0, parent)


In [ ]:
"""
Taller: Internals de un LLM estilo LLaMA
=================================================================
Instrucciones:
  - Busca los bloques marcados con TODO y completa el código.
  - Cada sección tiene una celda de verificación al final.
  - No modifiques nada fuera de los bloques TODO.

Secciones:
  3. GQA vs MHA — efecto de n_kv_heads
"""

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Optional

# Importamos el modelo de referencia para comparaciones
from src.model import (
    ModelConfig, RMSNorm, SwiGLUFFN, MiniLLaMA,
    precompute_rope_freqs, apply_rope, GroupedQueryAttention
)
from src.data import get_corpus
from src.tokenizer import BPETokenizer

In [ ]:
# ===========================================================================
# SECCIÓN 3 — GQA vs MHA
# ===========================================================================
# Multi-Head Attention (MHA): n_kv_heads == n_heads  (sin compartir)
# Grouped Query Attention (GQA): n_kv_heads < n_heads (KV compartidos)
#
# Experimenta cambiando n_kv_heads y observa:
#   - Diferencia en parámetros de K y V
#   - Diferencia en los mapas de atención
# ===========================================================================

def build_attention(n_heads: int, n_kv_heads: int, d_model: int = 32) -> GroupedQueryAttention:
    # TODO 3.1 — Construye un GroupedQueryAttention con los parámetros dados.
    # Usamos d_ff = d_model * 2 y max_seq_len genérico; sólo importa el módulo
    # de atención, no el modelo completo.
    cfg = ModelConfig(
        vocab_size  = 256,
        d_model     = d_model,
        n_heads     = n_heads,
        n_kv_heads  = n_kv_heads,
        d_ff        = d_model * 2,
        max_seq_len = 128,
    )
    return GroupedQueryAttention(cfg)


def compare_attention_maps(attn_mha, attn_gqa, x, rope_freqs, mask):
    # TODO 3.2 — Extrae los pesos de atención (scores softmax) de ambos módulos.
    # Replicamos el forward de GroupedQueryAttention hasta el F.softmax y
    # retornamos los tensores (B, H, T, T) — H es n_heads (cabezas de Q) en ambos.

    def _get_attn_weights(mod, x, rope_freqs, mask):
        B, T, D = x.shape
        Dh = mod.head_dim

        q = mod.Wq(x).reshape(B, T, mod.n_heads,    Dh)
        k = mod.Wk(x).reshape(B, T, mod.n_kv_heads, Dh)
        v = mod.Wv(x).reshape(B, T, mod.n_kv_heads, Dh)  # noqa: F841

        q = apply_rope(q, rope_freqs)
        k = apply_rope(k, rope_freqs)

        # Expand K to match Q heads (GQA broadcasting)
        k = k.unsqueeze(3).expand(B, T, mod.n_kv_heads, mod.n_rep, Dh)              .reshape(B, T, mod.n_heads, Dh)

        q = q.transpose(1, 2)   # (B, Hq, T, Dh)
        k = k.transpose(1, 2)   # (B, Hq, T, Dh)

        scale  = math.sqrt(Dh)
        scores = torch.matmul(q, k.transpose(-2, -1)) / scale  # (B, Hq, T, T)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        return F.softmax(scores, dim=-1)   # (B, Hq, T, T)

    attn_weights_mha = _get_attn_weights(attn_mha, x, rope_freqs, mask)
    attn_weights_gqa = _get_attn_weights(attn_gqa, x, rope_freqs, mask)
    return attn_weights_mha, attn_weights_gqa


# ── Verificación 3 ──────────────────────────────────────────────────────────
def verify_section3():
    print("=" * 55)
    print("VERIFICACIÓN 3 — GQA vs MHA")
    print("=" * 55)
    D, Hq, Hkv, T = 32, 4, 2, 10

    mha = build_attention(n_heads=Hq,  n_kv_heads=Hq,  d_model=D)
    gqa = build_attention(n_heads=Hq,  n_kv_heads=Hkv, d_model=D)

    # Contar parámetros K+V
    mha_kv = mha.Wk.weight.numel() + mha.Wv.weight.numel()
    gqa_kv = gqa.Wk.weight.numel() + gqa.Wv.weight.numel()
    print(f"  MHA  K+V params : {mha_kv:,}")
    print(f"  GQA  K+V params : {gqa_kv:,}")
    print(f"  Reducción       : {(1 - gqa_kv/mha_kv)*100:.0f}%  (esperado {(1-Hkv/Hq)*100:.0f}%)")

    # Shapes del forward
    cfg        = ModelConfig(vocab_size=256, d_model=D, n_heads=Hq,
                             n_kv_heads=Hkv, d_ff=64, max_seq_len=T)
    x          = torch.randn(1, T, D)
    rope_freqs = precompute_rope_freqs(D // Hq, T)
    mask       = torch.tril(torch.ones(T, T))

    out_mha = build_attention(Hq, Hq, D)(x, rope_freqs, mask)
    out_gqa = build_attention(Hq, Hkv, D)(x, rope_freqs, mask)
    print(f"  MHA output shape: {list(out_mha.shape)}  (esperado [1, {T}, {D}])")
    print(f"  GQA output shape: {list(out_gqa.shape)}  (esperado [1, {T}, {D}])")

    # Comparar mapas de atención
    w_mha, w_gqa = compare_attention_maps(mha, gqa, x, rope_freqs, mask)
    print(f"  MHA attn shape  : {list(w_mha.shape)}  (esperado [1, {Hq}, {T}, {T}])")
    print(f"  GQA attn shape  : {list(w_gqa.shape)}  (esperado [1, {Hq}, {T}, {T}])")

    shapes_ok = (out_mha.shape == out_gqa.shape == torch.Size([1, T, D]))
    kv_ok     = (gqa_kv == mha_kv * Hkv // Hq)
    if shapes_ok and kv_ok:
        print("\n  ✓ Sección 3 correcta")
    else:
        print("\n  ✗ Revisa tu implementación")

    # Visualización de mapas de atención (cabeza 0)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, w, title in zip(axes, [w_mha, w_gqa], ["MHA (head 0)", "GQA (head 0)"]):
        im = ax.imshow(w[0, 0].detach().numpy(), vmin=0, vmax=1, cmap="Blues")
        ax.set_title(title)
        ax.set_xlabel("Key position")
        ax.set_ylabel("Query position")
        plt.colorbar(im, ax=ax)
    plt.suptitle("Attention maps — MHA vs GQA")
    plt.tight_layout()
    plt.savefig("attention_maps.png", dpi=130)
    print("  Plot guardado en attention_maps.png")


In [ ]:
verify_section3()